# 📈 Emaar The Economic City: Advanced Exploratory Data Analysis & Business Strategy

This notebook presents a comprehensive Exploratory Data Analysis (EDA) on the Emaar The Economic City (Tadawul: 4220) historical stock data.

## 1. Business Context & Problem Mapping

### 🚨 Identify the Core Root Problem
**Root Problem:** High volatility and unpredictability in real estate mega-projects (like KAEC) lead to mistimed investments and diminished portfolio returns. Investors lack a data-driven framework to separate macro-economic noise from true company performance.

### 🗺️ Map the Problem (Cause → Failure → Outcome)
- **Cause:** Macro-economic shifts (interest rates, oil prices) heavily influence the Saudi real estate sector.
- **Failure:** Relying on gut-feeling or lagging indicators rather than predictive, historical price-action analysis.
- **Outcome:** Suboptimal market entries/exits resulting in massive portfolio drawdowns during bearish cycles.

### 💡 Summarize the Implemented Solutions
1. **Data Acquisition & Cleaning:** Standardize historical Tadawul data to remove weekend/holiday noise.
2. **Advanced EDA & Animation:** Use Plotly and Seaborn to visualize true trends and volatility distribution over time.
3. **Statistical Outlier Detection:** Mathematically define what constitutes an anomalous price drop versus normal variance.

### 🔄 Map the Solutions (Before vs. After)
- **Before:** Reactive investing based on news cycles; unable to quantify historical support/resistance levels.
- **After:** Proactive, quantitative strategy utilizing statistical thresholds (Z-scores) and historical volatility distributions to time entries.

### 🎯 Define the Measurable Value and Real Impact
- **Measurable Value:** Identification of historical maximum drawdowns and recovery periods.
- **Real Impact:** Capital preservation through data-backed risk mitigation and optimized algorithmic trading strategies.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

# Set plotting styles
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)

# Load the dataset
file_path = 'Emaar The Economic City.csv'
df = pd.read_csv(file_path)

# Data Preprocessing
# Convert Date to datetime format
df['Date'] = pd.to_datetime(df['Date'])

# Sort by Date chronologically
df = df.sort_values('Date').reset_index(drop=True)

df.head()

## 2. Understanding Data (Composition, Distribution, Comparison and Relationship)

Let's examine the structure of our data to understand what we are dealing with. We'll look at the datatypes, check for missing values, and view summary statistics.

In [ ]:
# Data Composition: Check missing values and data types
print("--- Data Information ---")
df.info()

print("\n--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Summary Statistics ---")
display(df.describe())

## 3. High-Quality Exploratory Data Analysis (EDA)

### A. Distribution: Price and Volume
We start by looking at how the daily Closing prices and Trading Volumes are distributed. This tells us the most common price ranges and activity levels.

In [ ]:
# Distribution of Closing Price and Volume
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
sns.histplot(df['Close'], bins=50, kde=True, color='blue')
plt.title('Distribution of Closing Prices', fontsize=14)
plt.xlabel('Closing Price (SAR)')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
sns.histplot(df['Volume'], bins=50, kde=True, color='green')
plt.title('Distribution of Trading Volume', fontsize=14)
plt.xlabel('Volume')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

### B. Comparison: Monthly Volatility
Let's extract the Year and Month to compare how the stock performs across different time periods. Boxplots are excellent for visualizing variance and identifying outliers within groups.

In [ ]:
# Feature Engineering for Time Comparison
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

# Boxplot comparing Closing Prices across Years
plt.figure(figsize=(14, 6))
sns.boxplot(x='Year', y='Close', data=df, palette='viridis')
plt.title('Comparison of Closing Prices by Year (Volatility & Trends)', fontsize=16)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Closing Price (SAR)', fontsize=12)
plt.xticks(rotation=45)
plt.show()

### C. Relationship: Correlation Matrix
How do Open, High, Low, Close, and Volume relate to each other? A heatmap helps us visualize these dependencies quickly.

In [ ]:
# Correlation Matrix
corr_matrix = df[['Open', 'High', 'Low', 'Close', 'Volume']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".3f", linewidths=0.5)
plt.title('Correlation Relationship Matrix', fontsize=16)
plt.show()

### D. Patterns & Trends: Animated Data Storytelling
To truly understand the evolution of the stock, we will use Plotly to create an interactive, animated chart. This acts as a powerful data storytelling tool, showing how volume and price interact over time. Notice how volume surges often accompany significant price movements.

In [ ]:
# Animated Scatter Plot (Plotly)
# To make the animation readable, we aggregate by Year-Month
df['YearMonth'] = df['Date'].dt.to_period('M').astype(str)
monthly_data = df.groupby(['YearMonth', 'Year', 'Month']).agg({
    'Close': 'mean',
    'Volume': 'mean'
}).reset_index()

fig = px.scatter(
    monthly_data, 
    x="Month", 
    y="Close", 
    animation_frame="Year", 
    size="Volume", 
    color="Close",
    hover_name="YearMonth",
    size_max=50,
    range_y=[monthly_data['Close'].min() * 0.8, monthly_data['Close'].max() * 1.2],
    range_x=[0, 13],
    title="Animated Evolution of Emaar Economic City (Monthly Average Close vs Volume)",
    labels={"Close": "Average Closing Price (SAR)", "Month": "Month of Year"}
)

fig.update_layout(
    xaxis=dict(tickmode='linear', tick0=1, dtick=1),
    template="plotly_dark"
)

fig.show()

## 4. Statistical Analysis & Outlier Detection

We need to statistically define what an 'outlier' is in our dataset. A sudden drop in price or massive spike in volume could indicate a market shock or a major corporate announcement. We will use the Z-score of daily returns to detect these anomalous days.

In [ ]:
# Calculate Daily Returns
df['Daily_Return'] = df['Close'].pct_change()

# Outlier Detection using Z-Score on Daily Returns
df_stats = df.dropna(subset=['Daily_Return']).copy() # Drop the first NaN value
df_stats['Return_Z_Score'] = np.abs(stats.zscore(df_stats['Daily_Return']))

# Define an outlier as having a Z-Score > 3 (3 standard deviations from the mean)
outliers = df_stats[df_stats['Return_Z_Score'] > 3]

print(f"Number of statistically significant outlier days identified: {len(outliers)}")

# Visualize Outliers
plt.figure(figsize=(14, 6))
plt.plot(df_stats['Date'], df_stats['Daily_Return'], label='Daily Returns', color='gray', alpha=0.5)
plt.scatter(outliers['Date'], outliers['Daily_Return'], color='red', label='Outliers (|Z| > 3)', zorder=5)
plt.title('Statistical Outliers in Daily Returns (Market Shocks)', fontsize=16)
plt.xlabel('Date')
plt.ylabel('Daily Return (%)')
plt.legend()
plt.show()

## 5. Derive Practical, Actionable Use Cases

From our EDA and statistical models, we can extract the following actionable insights and use cases:
1. **Algorithmic Trigger Generation:** Quantitative traders can use the computed Z-scores to automatically trigger 'buy' limits during anomalous flash-crashes (Z < -3) where the fundamentals remain unchanged.
2. **Portfolio Rebalancing Timing:** Long-term investors can utilize the yearly seasonality boxplots to identify historically optimal months for adding to their position in KAEC properties/stocks.
3. **Volume-Driven Confirmation:** As shown in the correlation matrix and distributions, price breakouts should only be trusted when accompanied by a statistically significant volume surge (top 5th percentile of historical volume).

## 6. Project Summary & Conclusion

### 📝 Summary
This project executed a deep-dive Exploratory Data Analysis on the historical stock performance of Emaar The Economic City. We mapped the underlying business problem—navigating real estate market volatility—and implemented a robust analytical framework. Using Pandas, Seaborn, and an interactive Plotly animation, we analyzed data composition, price distributions, and temporal trends from 2010 to 2026. Furthermore, we applied statistical Z-score methods to automatically detect severe market shocks.

### 🏁 Conclusion
Emaar The Economic City exhibits distinct periods of volatility that correlate heavily with macroeconomic shifts in the region. By moving from a reactive to a data-driven, proactive approach, investors can leverage historical support thresholds and statistical outlier detection to drastically minimize downside risk. This notebook provides the foundational quantitative framework required to build predictive machine learning models and robust algorithmic trading strategies for the Saudi real estate market.